# Exploratory Data Analysis (EDA)

We perform Exploratory Data Analysis (EDA) and upload our results to Weights & Biases.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbs
import wandb
import wandb_utils # noqa: F401 # we need the Artifact.file work-around

from data_profiling import ProfileReport
# ydata-profiling recently got renamed to fg-data-profiling
# functionality stays the same though

# Parameters

Project and group have to be set either via `WANDB_PROJECT` and `WANDB_RUN_GROUP` environment variables or Jupyter notebook parameters (e.g. via Papermill).

In [ ]:
wandb_project = os.environ.get('WANDB_PROJECT', None)
wandb_group = os.environ.get('WANDB_RUN_GROUP', None)

In [ ]:
# abort this notebook if W&B environment variables are missing
if any(x is None for x in (wandb_project, wandb_group)):
    raise RuntimeError('WANDB_PROJECT and WANDB_RUN_GROUP environment variables must be set!')

## Weights & Biases

We retrieve the initial `sample.csv` artifact from Weights & Biases.

In [ ]:
# create run and download artifact
run = wandb.init(project=wandb_project, group=wandb_group, job_type='eda', save_code=True)

filename = run.use_artifact('sample.csv:v0').file()
df = pd.read_csv(filename, low_memory=False)

## Data Visualization

First, show the raw data as a table to get some feel for it, then create an interactive report.

In [ ]:
df.describe()

In [ ]:
df.head()

### Profile Report

In [ ]:
# create profiling report
profile = ProfileReport(df, title="YData Profiling Report", explorative=True)
# use to_notebook_iframe instead of to_widgets, because of ValueError: ('widget type not understood', 'overview_tabs')
# see https://github.com/Data-Centric-AI-Community/fg-data-profiling/issues/1763
profile.to_notebook_iframe()

### Price Analysis

Let's create a series of targeted histograms of the price ranges. We include a kernel density function for the finer graphs only. Plot all prices on a logarithmic scale to get some idea of the outliers on the high side.

In [ ]:
%matplotlib inline

# limit our plot to mean +/- 1 std to get a better look
mean_price, std_price = df['price'].mean(), df['price'].std()
min_price, max_price = mean_price - 1 * std_price, mean_price + 1 * std_price

plt.figure(figsize=(15, 5))

# Figure 1: Price Distribution
plt.subplot(1, 3, 1)
sbs.histplot(df[df['price'].between(min_price, max_price)], x='price', bins=30, kde=True)
plt.title('Temporary Rental Price Distribution (μ ± 1σ)')
plt.xlabel('Price Per Night / USD')
plt.xticks(np.arange(0, max_price + 25, 25), rotation=90);
plt.xlim(min_price, max_price);

# Figure 2: Low Prices
max_price = 100
plt.subplot(1, 3, 2)
sbs.histplot(df[df['price'] < max_price], x='price', bins=30, kde=True)
plt.title(f'Temporary Rental Price Distribution (< {max_price}$)')
plt.xlabel('Price Per Night / USD')
plt.xticks(np.arange(0, max_price + 10, 10), rotation=90);
plt.xlim(0, max_price);

# Figure 3: Extremes
plt.subplot(1, 3, 3)
sbs.histplot(df, x='price', bins=100, zorder=3)
plt.title('Temporary Rental Price Distribution (All Prices)')
plt.xlabel('Price Per Night / USD')
plt.xticks(np.arange(0, df['price'].max() + 1000, 1000), rotation=90)
plt.yscale('log')
plt.ylabel('Count (log scale)');
plt.grid(axis='y')
plt.tight_layout()

## Data Cleanup

We notice that we need to limit price range, because we have some extremely cheap and some incredibly expensive `price` entries. We also notice that the `last_review` column is really a date time.

In [ ]:
# Drop Price Outliers

min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

In [ ]:
df.info()

## Finish

In [ ]:
run.finish()